# exp052 Stage 1: XC Backbone Pretrain (Colab Blackwell)

**目的**: paper 256 流 XC pretrain (paper 256 評価 +0.046 LB の simplified version)。

## Data

| Source | files | size |
|---|---|---|
| Kaggle Dataset Part 1 | 13,399 mp3 | 17.6 GB |
| Kaggle Dataset Part 2 | 12,978 mp3 | 15.3 GB |
| **合計** | **26,377** | **32.9 GB** |

カバー: 159 BC2026 Aves species (162 中 -3 = XC 未登録)

## Design (paper 256 準拠)

| Item | Value |
|---|---|
| Backbone | `tf_efficientnetv2_s.in21k_ft_in1k` |
| Output | **234-class multi-label** (BC2026 全 taxonomy) |
| Loss | BCEWithLogitsLoss |
| Input | 5s random chunk, mel n_mels=128, sr=32000 |
| Aug | Spec mixup α=0.5 + SpecAugment + waveform mixup α=0.5 |
| Optimizer | AdamW lr=1e-3 wd=1e-4 (warmup 500 step) |
| Scheduler | CosineAnnealingLR T_max=30 |
| Epochs | **30** |
| Batch | **256** (Blackwell 96GB) |
| Label smoothing | 0.05 |

## Output

- Backbone state_dict to Drive (`stage1_backbone.pth`)
- history.json with rich log per epoch

## 期待

- Aves AUC 0.90-0.95 (BC train_audio より高密度な XC で学習)
- Backbone audio domain adapted、Stage 2 finetune 効率良
- Total time on Blackwell: **2.5-3.5h**


In [1]:
# ============================================================
# Cell 1: Install + Drive mount + Kaggle API setup
# ============================================================
!pip install -q timm kaggle soundfile librosa torchaudio tqdm

from google.colab import drive
drive.mount("/content/drive")

import os, json, shutil
from pathlib import Path

# Drive paths
DRIVE_ROOT = Path("/content/drive/MyDrive/kaggle/birdclef2026/exp052")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_DIR = DRIVE_ROOT / "ckpt"
CKPT_DIR.mkdir(exist_ok=True)

# Kaggle credentials - try multiple locations
KAGGLE_JSON_CANDIDATES = [
    Path("/content/drive/MyDrive/kaggle/birdclef2026/kaggle.json"),  # ★ 標準
    Path("/content/drive/MyDrive/kaggle.json"),
    Path("/content/drive/MyDrive/.kaggle/kaggle.json"),
    Path("/content/kaggle.json"),
    Path.home() / ".kaggle" / "kaggle.json",
]
KAGGLE_JSON_PATH = next((p for p in KAGGLE_JSON_CANDIDATES if p.exists()), None)

if KAGGLE_JSON_PATH is None:
    print("kaggle.json not found in:")
    for p in KAGGLE_JSON_CANDIDATES:
        print(f"  - {p}")
    print("\nUpload kaggle.json now:")
    from google.colab import files
    uploaded = files.upload()
    if "kaggle.json" in uploaded:
        target = Path("/content/drive/MyDrive/kaggle/birdclef2026/kaggle.json")
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(uploaded["kaggle.json"])
        KAGGLE_JSON_PATH = target
        print(f"OK Saved to {target}")
    else:
        raise FileNotFoundError("kaggle.json not uploaded")

print(f"Using kaggle.json from: {KAGGLE_JSON_PATH}")

# ★ KGAT_ Bearer token を KAGGLE_API_TOKEN env var に設定 (CLAUDE.md memory)
# Basic auth (kaggle.json 直接) だと 401/403 になる
creds = json.loads(KAGGLE_JSON_PATH.read_text())
if "key" in creds:
    token = creds["key"]
    os.environ["KAGGLE_API_TOKEN"] = token
    print(f"OK KAGGLE_API_TOKEN env var set (token starts: {token[:10]}...)")
else:
    raise ValueError(f"kaggle.json missing 'key' field: {creds.keys()}")

# Also copy to ~/.kaggle/ for backup (CLI fall back behavior)
os.makedirs(Path.home() / ".kaggle", exist_ok=True)
shutil.copy(KAGGLE_JSON_PATH, Path.home() / ".kaggle" / "kaggle.json")
os.chmod(Path.home() / ".kaggle" / "kaggle.json", 0o600)

# Verify Kaggle API (should NOT 403)
print("\nTesting Kaggle API...")
import subprocess
r = subprocess.run("kaggle datasets list -m 1", shell=True, capture_output=True, text=True,
                   env={**os.environ})
print(r.stdout[:500])
if "403" in r.stderr or "401" in r.stderr or "Forbidden" in r.stderr:
    print(f"⚠ Auth error: {r.stderr[:500]}")
    raise RuntimeError("Kaggle API auth failed - check KGAT token in kaggle.json")
else:
    print("OK Kaggle API ready")


Mounted at /content/drive
Using kaggle.json from: /content/drive/MyDrive/kaggle/birdclef2026/kaggle.json
OK KAGGLE_API_TOKEN env var set (token starts: KGAT_bb340...)

Testing Kaggle API...

OK Kaggle API ready


In [2]:
# ============================================================
# Cell 2: Download XC Datasets via SDK (NOT CLI subprocess)
# ============================================================
# Use kaggle SDK directly to avoid CLI subprocess env var issues.
# SDK respects KAGGLE_API_TOKEN env var for Bearer tokens (KGAT_).

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()
print("OK Kaggle SDK authenticated")

DATA_ROOT = Path("/content/xc_data")
DATA_ROOT.mkdir(exist_ok=True)
part1_dir = DATA_ROOT / "part1"
part2_dir = DATA_ROOT / "part2"

def sdk_download(slug, target_dir, expected_subdir):
    target_dir.mkdir(exist_ok=True, parents=True)
    expected_meta = target_dir / expected_subdir / "_metadata.csv"
    if expected_meta.exists():
        print(f"  Already exists: {expected_meta}")
        return True
    print(f"\nDownloading {slug} -> {target_dir} (this takes ~5-10 min for 17 GB)")
    try:
        api.dataset_download_files(slug, path=str(target_dir), unzip=True, quiet=False)
    except Exception as e:
        print(f"FAIL: {str(e)[:300]}")
        raise
    if expected_meta.exists():
        print(f"  OK metadata found: {expected_meta}")
        return True
    print(f"  WARN: expected metadata not found at {expected_meta}, listing dir:")
    for p in target_dir.rglob("*.csv"):
        print(f"    {p}")
    raise FileNotFoundError(f"Expected {expected_meta} after extract")

sdk_download("maekeso/birdclef2026-xc-api-dl-part1", part1_dir, "xc_api")
sdk_download("maekeso/birdclef2026-xc-api-dl-part2", part2_dir, "xc_api_part2")

# Verify
import pandas as pd
p1_meta = part1_dir / "xc_api" / "_metadata.csv"
p2_meta = part2_dir / "xc_api_part2" / "_metadata.csv"
p1_df = pd.read_csv(p1_meta)
p2_df = pd.read_csv(p2_meta)
print(f"\n=== Datasets ready ===")
print(f"Part 1: {len(p1_df)} files, {p1_df['scientific_name'].nunique()} species")
print(f"Part 2: {len(p2_df)} files, {p2_df['scientific_name'].nunique()} species")

# Sample audio file check
sample_mp3 = next(part1_dir.rglob("*.mp3"), None)
if sample_mp3:
    print(f"\nSample mp3: {sample_mp3} ({sample_mp3.stat().st_size/1e6:.1f} MB)")


OK Kaggle SDK authenticated

Dataset URL: https://www.kaggle.com/datasets/maekeso/birdclef2026-xc-api-dl-part1


100%|██████████| 14.5G/14.5G [06:07<00:00, 42.3MB/s]



  OK metadata found: /content/xc_data/part1/xc_api/_metadata.csv

Dataset URL: https://www.kaggle.com/datasets/maekeso/birdclef2026-xc-api-dl-part2


100%|██████████| 12.5G/12.5G [05:18<00:00, 42.0MB/s]



  OK metadata found: /content/xc_data/part2/xc_api_part2/_metadata.csv

=== Datasets ready ===
Part 1: 13399 files, 77 species
Part 2: 12978 files, 83 species

Sample mp3: /content/xc_data/part1/xc_api/audio/Sittasomus_griseicapillus/XC126002.mp3 (0.1 MB)


In [3]:
# ============================================================
# Cell 3: Setup torch + GPU check
# ============================================================
import os, sys, json, time, math, random, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import timm
import soundfile as sf
import librosa
from tqdm.auto import tqdm

print(f"torch: {torch.__version__}")
print(f"cuda: {torch.cuda.is_available()}, devs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name} ({p.total_memory/1e9:.1f} GB)")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


torch: 2.10.0+cu128
cuda: True, devs: 1
  [0] NVIDIA RTX PRO 6000 Blackwell Server Edition (102.0 GB)


In [4]:
# ============================================================
# Cell 4: BC2026 species list (embedded, 234 species)
# ============================================================
__BC2026_SPECIES = [
    ('Guyalna cuta', '1161364', 'Insecta'),
    ('Caiman yacare', '116570', 'Reptilia'),
    ('Leptodactylus luctator', '1176823', 'Amphibia'),
    ('Adenomera guarani', '1491113', 'Amphibia'),
    ('Lysapsus limellum', '1595929', 'Amphibia'),
    ('Equus caballus', '209233', 'Mammalia'),
    ('Leptodactylus syphax', '22930', 'Amphibia'),
    ('Leptodactylus mystacinus', '22956', 'Amphibia'),
    ('Leptodactylus podicipinus', '22961', 'Amphibia'),
    ('Leptodactylus elenae', '22967', 'Amphibia'),
    ('Leptodactylus fuscus', '22973', 'Amphibia'),
    ('Leptodactylus labyrinthicus', '22983', 'Amphibia'),
    ('Leptodactylus petersii', '22985', 'Amphibia'),
    ('Physalaemus centralis', '23150', 'Amphibia'),
    ('Physalaemus albifrons', '23154', 'Amphibia'),
    ('Physalaemus albonotatus', '23158', 'Amphibia'),
    ('Pseudopaludicola mystacalis', '23176', 'Amphibia'),
    ('Phyllomedusa sauvagii', '23724', 'Amphibia'),
    ('Scinax nasicus', '24279', 'Amphibia'),
    ('Scinax fuscovarius', '24285', 'Amphibia'),
    ('Scinax fuscomarginatus', '24287', 'Amphibia'),
    ('Scinax acuminatus', '24321', 'Amphibia'),
    ('Quesada gigas', '244024', 'Insecta'),
    ('Chiasmocleis mehelyi', '25073', 'Amphibia'),
    ('Elachistocleis bicolor', '25092', 'Amphibia'),
    ('Dermatonotus muelleri', '25214', 'Amphibia'),
    ('Physalaemus biligonigerus', '326272', 'Amphibia'),
    ('Panthera onca', '41970', 'Mammalia'),
    ('Alouatta caraya', '43435', 'Mammalia'),
    ('Canis familiaris', '47144', 'Mammalia'),
    ('Insect son01', '47158son01', 'Insecta'),
    ('Insect son02', '47158son02', 'Insecta'),
    ('Insect son03', '47158son03', 'Insecta'),
    ('Insect son04', '47158son04', 'Insecta'),
    ('Insect son05', '47158son05', 'Insecta'),
    ('Insect son06', '47158son06', 'Insecta'),
    ('Insect son07', '47158son07', 'Insecta'),
    ('Insect son08', '47158son08', 'Insecta'),
    ('Insect son09', '47158son09', 'Insecta'),
    ('Insect son10', '47158son10', 'Insecta'),
    ('Insect son11', '47158son11', 'Insecta'),
    ('Insect son12', '47158son12', 'Insecta'),
    ('Insect son13', '47158son13', 'Insecta'),
    ('Insect son14', '47158son14', 'Insecta'),
    ('Insect son15', '47158son15', 'Insecta'),
    ('Insect son16', '47158son16', 'Insecta'),
    ('Insect son17', '47158son17', 'Insecta'),
    ('Insect son18', '47158son18', 'Insecta'),
    ('Insect son19', '47158son19', 'Insecta'),
    ('Insect son20', '47158son20', 'Insecta'),
    ('Insect son21', '47158son21', 'Insecta'),
    ('Insect son22', '47158son22', 'Insecta'),
    ('Insect son23', '47158son23', 'Insecta'),
    ('Insect son24', '47158son24', 'Insecta'),
    ('Insect son25', '47158son25', 'Insecta'),
    ('Physalaemus nattereri', '476521', 'Amphibia'),
    ('Sapajus cay', '516975', 'Mammalia'),
    ('Pithecopus azureus', '517063', 'Amphibia'),
    ('Boana lundii', '555123', 'Amphibia'),
    ('Boana punctata', '555145', 'Amphibia'),
    ('Boana raniceps', '555146', 'Amphibia'),
    ('Ameerega picta', '64898', 'Amphibia'),
    ('Dendropsophus minutus', '65377', 'Amphibia'),
    ('Dendropsophus nanus', '65380', 'Amphibia'),
    ('Pseudis platensis', '66971', 'Amphibia'),
    ('Rhinella diptycha', '67107', 'Amphibia'),
    ('Trachycephalus typhonius', '67252', 'Amphibia'),
    ('Leptodactylus macrosternum', '70711', 'Amphibia'),
    ('Plecturocebus pallescens', '738183', 'Mammalia'),
    ('Bos taurus', '74113', 'Mammalia'),
    ('Mico melanurus', '74580', 'Mammalia'),
    ('Prionacris erosa', '760266', 'Insecta'),
    ('Hylophilus pectoralis', 'ashgre1', 'Aves'),
    ('Mustelirallus albicollis', 'astcra1', 'Aves'),
    ('Crax fasciolata', 'bafcur1', 'Aves'),
    ('Micrastur ruficollis', 'baffal1', 'Aves'),
    ('Coereba flaveola', 'banana', 'Aves'),
    ('Thamnophilus doliatus', 'barant1', 'Aves'),
    ('Procnias nudicollis', 'batbel1', 'Aves'),
    ('Ara ararauna', 'baymac', 'Aves'),
    ('Dendrocygna autumnalis', 'bbwduc', 'Aves'),
    ('Microspingus melanoleucus', 'bcwfin2', 'Aves'),
    ('Donacobius atricapilla', 'bkcdon', 'Aves'),
    ('Aratinga nenday', 'bkhpar', 'Aves'),
    ('Busarellus nigricollis', 'blchaw1', 'Aves'),
    ('Spizaetus tyrannus', 'blheag1', 'Aves'),
    ('Tityra cayana', 'blttit1', 'Aves'),
    ('Myiarchus tyrannulus', 'bncfly', 'Aves'),
    ('Megarynchus pitangua', 'bobfly1', 'Aves'),
    ('Progne tapera', 'brcmar1', 'Aves'),
    ('Tyto furcata', 'brnowl', 'Aves'),
    ('Momotus momota', 'bucmot4', 'Aves'),
    ('Thectocercus acuticaudatus', 'bucpar', 'Aves'),
    ('Amazona aestiva', 'bufpar', 'Aves'),
    ('Theristicus caudatus', 'bunibi1', 'Aves'),
    ('Athene cunicularia', 'burowl', 'Aves'),
    ('Colaptes campestris', 'camfli1', 'Aves'),
    ('Ortalis canicollis', 'chacha1', 'Aves'),
    ('Mimus saturninus', 'chbmoc1', 'Aves'),
    ('Gnorimopsar chopi', 'chobla1', 'Aves'),
    ('Conirostrum speciosum', 'chvcon1', 'Aves'),
    ('Synallaxis hypospodia', 'cibspi1', 'Aves'),
    ('Micrastur semitorquatus', 'coffal1', 'Aves'),
    ('Nyctidromus albicollis', 'compau', 'Aves'),
    ('Nyctibius griseus', 'compot1', 'Aves'),
    ('Turdus amaurochalinus', 'crbthr1', 'Aves'),
    ('Pachyramphus validus', 'crebec1', 'Aves'),
    ('Taoniscus nanus', 'dwatin1', 'Aves'),
    ('Icterus pyrrhopterus', 'epaori4', 'Aves'),
    ('Lathrotriccus euleri', 'eulfly1', 'Aves'),
    ('Cantorchilus guarayanus', 'fabwre1', 'Aves'),
    ('Glaucidium brasilianum', 'fepowl', 'Aves'),
    ('Machaeropterus pyrocephalus', 'ficman1', 'Aves'),
    ('Myiothlypis flaveola', 'flawar1', 'Aves'),
    ('Tyrannus savana', 'fotfly', 'Aves'),
    ('Cnemotriccus fuscatus', 'fusfly1', 'Aves'),
    ('Hylocharis chrysura', 'gilhum1', 'Aves'),
    ('Aramides ypecaha', 'giwrai1', 'Aves'),
    ('Chionomesa fimbriata', 'glteme1', 'Aves'),
    ('Saltator coerulescens', 'grasal3', 'Aves'),
    ('Crotophaga major', 'greani1', 'Aves'),
    ('Taraba major', 'greant1', 'Aves'),
    ('Myiopagis viridicata', 'greela', 'Aves'),
    ('Pitangus sulphuratus', 'grekis', 'Aves'),
    ('Nyctibius grandis', 'grepot1', 'Aves'),
    ('Phacellodomus ruber', 'gretho2', 'Aves'),
    ('Tringa melanoleuca', 'greyel', 'Aves'),
    ('Leptotila rufaxilla', 'grfdov1', 'Aves'),
    ('Eucometis penicillata', 'grhtan1', 'Aves'),
    ('Aramides cajaneus', 'gycwor1', 'Aves'),
    ('Anhima cornuta', 'horscr1', 'Aves'),
    ('Passer domesticus', 'houspa', 'Aves'),
    ('Anodorhynchus hyacinthinus', 'hyamac1', 'Aves'),
    ('Elaenia spectabilis', 'larela1', 'Aves'),
    ('Elaenia chiriquensis', 'lesela1', 'Aves'),
    ('Emberizoides ypiranganus', 'lesgrf1', 'Aves'),
    ('Aramus guarauna', 'limpki', 'Aves'),
    ('Dryocopus lineatus', 'linwoo1', 'Aves'),
    ('Coccycua minuta', 'litcuc2', 'Aves'),
    ('Setopagis parvula', 'litnig1', 'Aves'),
    ('Pyrrhura frontalis', 'mabpar', 'Aves'),
    ('Cercomacra melanaria', 'magant1', 'Aves'),
    ('Cissopis leverianus', 'magtan2', 'Aves'),
    ('Polioptila dumicola', 'masgna1', 'Aves'),
    ('Chordeiles nacunda', 'nacnig1', 'Aves'),
    ('Rufirallus schomburgkii', 'ocecra1', 'Aves'),
    ('Sittasomus griseicapillus', 'oliwoo1', 'Aves'),
    ('Icterus croconotus', 'orbtro3', 'Aves'),
    ('Amazona amazonica', 'orwpar', 'Aves'),
    ('Pandion haliaetus', 'osprey', 'Aves'),
    ('Synallaxis albescens', 'pabspi1', 'Aves'),
    ('Furnarius leucopus', 'palhor3', 'Aves'),
    ('Thraupis palmarum', 'paltan1', 'Aves'),
    ('Dromococcyx phasianellus', 'phecuc1', 'Aves'),
    ('Patagioenas picazuro', 'picpig2', 'Aves'),
    ('Legatus leucophaius', 'pirfly1', 'Aves'),
    ('Thamnophilus pelzelni', 'plasla1', 'Aves'),
    ('Inezia inornata', 'platyr1', 'Aves'),
    ('Cyanocorax chrysops', 'plcjay1', 'Aves'),
    ('Theristicus caerulescens', 'pluibi1', 'Aves'),
    ('Cyanocorax cyanomelas', 'purjay1', 'Aves'),
    ('Hemitriccus margaritaceiventer', 'pvttyr1', 'Aves'),
    ('Ara chloropterus', 'ragmac1', 'Aves'),
    ('Campylorhamphus trochilirostris', 'rebscy1', 'Aves'),
    ('Coryphospingus cucullatus', 'recfin1', 'Aves'),
    ('Gallus gallus', 'redjun', 'Aves'),
    ('Cariama cristata', 'relser1', 'Aves'),
    ('Megaceryle torquata', 'rinkin1', 'Aves'),
    ('Myiothlypis rivularis', 'rivwar1', 'Aves'),
    ('Rupornis magnirostris', 'roahaw', 'Aves'),
    ('Turdus rufiventris', 'rubthr1', 'Aves'),
    ('Pseudoseisura unirufa', 'rufcac2', 'Aves'),
    ('Casiornis rufus', 'rufcas2', 'Aves'),
    ('Conopophaga lineata', 'rufgna3', 'Aves'),
    ('Furnarius rufus', 'rufhor2', 'Aves'),
    ('Antrostomus rufus', 'rufnig1', 'Aves'),
    ('Phacellodomus rufifrons', 'ruftho1', 'Aves'),
    ('Poecilotriccus latirostris', 'ruftof1', 'Aves'),
    ('Myiozetetes cayanensis', 'rumfly1', 'Aves'),
    ('Tigrisoma lineatum', 'ruther1', 'Aves'),
    ('Galbula ruficauda', 'rutjac1', 'Aves'),
    ('Arremon flavirostris', 'sabspa1', 'Aves'),
    ('Sicalis flaveola', 'saffin', 'Aves'),
    ('Thraupis sayaca', 'saytan1', 'Aves'),
    ('Columbina squammata', 'scadov1', 'Aves'),
    ('Pionus maximiliani', 'schpar1', 'Aves'),
    ('Phaethornis eurynome', 'scther1', 'Aves'),
    ('Myiarchus ferox', 'shcfly1', 'Aves'),
    ('Accipiter striatus', 'shshaw', 'Aves'),
    ('Lurocalis semitorquatus', 'shtnig1', 'Aves'),
    ('Ramphocelus carbo', 'sibtan2', 'Aves'),
    ('Crotophaga ani', 'smbani', 'Aves'),
    ('Crypturellus parvirostris', 'smbtin1', 'Aves'),
    ('Cacicus solitarius', 'sobcac1', 'Aves'),
    ('Camptostoma obsoletum', 'sobtyr1', 'Aves'),
    ('Myiozetetes similis', 'socfly1', 'Aves'),
    ('Synallaxis frontalis', 'sofspi1', 'Aves'),
    ('Corythopis delalandi', 'souant1', 'Aves'),
    ('Vanellus chilensis', 'soulap1', 'Aves'),
    ('Chauna torquata', 'souscr1', 'Aves'),
    ('Hypoedaleus guttatus', 'spbant3', 'Aves'),
    ('Synallaxis spixi', 'spispi1', 'Aves'),
    ('Antiurus maculicaudus', 'sptnig1', 'Aves'),
    ('Piaya cayana', 'squcuc1', 'Aves'),
    ('Dendroplex picus', 'stbwoo2', 'Aves'),
    ('Tapera naevia', 'strcuc1', 'Aves'),
    ('Butorides striata', 'strher2', 'Aves'),
    ('Asio clamator', 'strowl1', 'Aves'),
    ('Eupetomena macroura', 'swthum1', 'Aves'),
    ('Chiroxiphia caudata', 'swtman1', 'Aves'),
    ('Crypturellus tataupa', 'tattin1', 'Aves'),
    ('Campylorhynchus turdinus', 'thlwre1', 'Aves'),
    ('Ramphastos toco', 'toctou1', 'Aves'),
    ('Tyrannus melancholicus', 'trokin', 'Aves'),
    ('Megascops choliba', 'trsowl', 'Aves'),
    ('Crypturellus undulatus', 'undtin1', 'Aves'),
    ('Thamnophilus caerulescens', 'varant1', 'Aves'),
    ('Jacana jacana', 'watjac1', 'Aves'),
    ('Pyriglena maura', 'wesfie1', 'Aves'),
    ('Dendrocygna viduata', 'wfwduc1', 'Aves'),
    ('Biatas nigropectus', 'whbant2', 'Aves'),
    ('Myiothlypis leucoblephara', 'whbwar2', 'Aves'),
    ('Melanerpes candidus', 'whiwoo1', 'Aves'),
    ('Synallaxis albilora', 'whlspi1', 'Aves'),
    ('Cyanocorax cyanopogon', 'whnjay1', 'Aves'),
    ('Leptotila verreauxi', 'whtdov', 'Aves'),
    ('Picumnus albosquamatus', 'whwpic1', 'Aves'),
    ('Caracara plancus', 'y00678', 'Aves'),
    ('Paroaria capitata', 'yebcar', 'Aves'),
    ('Elaenia flavogaster', 'yebela1', 'Aves'),
    ('Primolius auricollis', 'yecmac', 'Aves'),
    ('Brotogeris chiriri', 'yecpar', 'Aves'),
    ('Daptrius chimachima', 'yehcar1', 'Aves'),
    ('Tolmomyias sulphurescens', 'yeofly1', 'Aves'),
]

species_df = pd.DataFrame(__BC2026_SPECIES, columns=["scientific_name", "primary_label", "class_name"])
PRIMARY_LABELS = species_df["primary_label"].tolist()
SCI_NAME_LC_TO_LABEL = {str(s).lower(): l for s, l in zip(species_df['scientific_name'], species_df['primary_label'])}
LABEL_TO_IDX = {l: i for i, l in enumerate(PRIMARY_LABELS)}
LABEL_TO_CLASS = dict(zip(species_df['primary_label'], species_df['class_name']))

N_CLASSES = len(PRIMARY_LABELS)
print(f"BC2026 species: {N_CLASSES}")
print(f"  class breakdown: {species_df['class_name'].value_counts().to_dict()}")


BC2026 species: 234
  class breakdown: {'Aves': 162, 'Amphibia': 35, 'Insecta': 28, 'Mammalia': 8, 'Reptilia': 1}


In [5]:
# ============================================================
# Cell 5: Config
# ============================================================
class CFG:
    SEED = 42
    BACKBONE = "tf_efficientnetv2_s.in21k_ft_in1k"
    N_CLASSES = N_CLASSES  # 234

    # Audio
    SR = 32000
    CHUNK_SEC = 5
    CHUNK_LEN = SR * CHUNK_SEC

    # Mel
    N_MELS = 128
    N_FFT = 2048
    HOP = 512
    FMIN = 20
    FMAX = 16000

    # Train
    EPOCHS = 30
    BATCH_SIZE = 256  # Blackwell 96GB easy
    LR = 5e-4  # was 1e-3, safer (避 initial NaN)
    WD = 1e-4
    NUM_WORKERS = 4  # was 8, Colab CPU safety
    WARMUP_STEPS = 500

    # Aug
    MIXUP_PROB = 0.5
    MIXUP_ALPHA = 1.0  # was 0.5, Beta(1,1) uniform (less aggressive)
    WAVE_MIXUP_PROB = 0.3
    FREQ_MASK = 30
    TIME_MASK = 40
    LABEL_SMOOTH = 0.05

    # Output
    CKPT_DIR = CKPT_DIR
    BEST_CKPT = CKPT_DIR / "stage1_backbone_best.pth"
    LAST_CKPT = CKPT_DIR / "stage1_backbone_last.pth"
    HIST_JSON = CKPT_DIR / "stage1_history.json"

torch.manual_seed(CFG.SEED)
np.random.seed(CFG.SEED)
random.seed(CFG.SEED)
torch.backends.cudnn.benchmark = True
print(f"Backbone: {CFG.BACKBONE}")
print(f"Train: {CFG.EPOCHS} ep × batch {CFG.BATCH_SIZE}")
print(f"Output: {CFG.CKPT_DIR}")


Backbone: tf_efficientnetv2_s.in21k_ft_in1k
Train: 30 ep × batch 256
Output: /content/drive/MyDrive/kaggle/birdclef2026/exp052/ckpt


In [6]:
# ============================================================
# Cell 6: Build dataset metadata (merge Part 1 + Part 2)
# ============================================================
# Build (audio_path, primary_label, secondary_labels) tuples
records = []

for meta_path, audio_root in [
    (part1_dir / "xc_api" / "_metadata.csv", part1_dir / "xc_api" / "audio"),
    (part2_dir / "xc_api_part2" / "_metadata.csv", part2_dir / "xc_api_part2" / "audio"),
]:
    if not meta_path.exists():
        print(f"SKIP {meta_path} (not found)")
        continue
    df = pd.read_csv(meta_path)
    print(f"\n[{meta_path.parent.name}] {len(df)} files")

    for _, r in df.iterrows():
        sci = str(r.get("scientific_name", "")).strip()
        label = SCI_NAME_LC_TO_LABEL.get(sci.lower())
        if label is None:
            continue
        filename = r.get("filename", f"{sci.replace(' ', '_')}/XC{r['xc_id']}.mp3")
        audio_path = audio_root / filename
        # Try alt path (in case 'filename' isn't a full relative path)
        if not audio_path.exists():
            audio_path = audio_root / f"{sci.replace(' ', '_')}/XC{r['xc_id']}.mp3"
        if not audio_path.exists():
            continue
        records.append({
            "audio_path": str(audio_path),
            "primary_label": label,
            "scientific_name": sci,
            "duration": r.get("length_sec", 30),
        })

train_df = pd.DataFrame(records)
print(f"\nTotal pretrain records: {len(train_df)}")
print(f"Unique species: {train_df['primary_label'].nunique()} / {N_CLASSES}")
print(f"Per-species recordings (sample top 5):")
print(train_df['primary_label'].value_counts().head())



[xc_api] 13399 files

[xc_api_part2] 12978 files

Total pretrain records: 26377
Unique species: 159 / 234
Per-species recordings (sample top 5):
primary_label
houspa     2089
grekis      592
banana      506
roahaw      475
yeofly1     471
Name: count, dtype: int64


In [7]:
# ============================================================
# Cell 7: Dataset class (random 5s crop + multi-label)
# ============================================================
class XCDataset(Dataset):
    def __init__(self, df, sr=CFG.SR, chunk_len=CFG.CHUNK_LEN, training=True,
                 label_smooth=CFG.LABEL_SMOOTH):
        self.df = df.reset_index(drop=True)
        self.sr = sr
        self.chunk_len = chunk_len
        self.training = training
        self.label_smooth = label_smooth

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = row["audio_path"]
        try:
            with sf.SoundFile(audio_path) as f:
                sr = f.samplerate
                src_chunk = int(self.chunk_len * sr / self.sr)
                total = f.frames
                if total > src_chunk:
                    if self.training:
                        start = np.random.randint(0, total - src_chunk + 1)
                    else:
                        start = (total - src_chunk) // 2
                    f.seek(start)
                    wav = f.read(src_chunk, dtype="float32")
                else:
                    wav = f.read(dtype="float32")
            if wav.ndim > 1:
                wav = wav.mean(axis=1)
            if sr != self.sr:
                wav = librosa.resample(wav, orig_sr=sr, target_sr=self.sr)
        except Exception:
            wav = np.zeros(self.chunk_len, dtype=np.float32)

        if len(wav) >= self.chunk_len:
            wav = wav[:self.chunk_len]
        else:
            wav = np.pad(wav, (0, self.chunk_len - len(wav)))

        # Multi-label with label smoothing
        label = np.full(N_CLASSES, self.label_smooth / 2, dtype=np.float32)
        primary = row["primary_label"]
        if primary in LABEL_TO_IDX:
            label[LABEL_TO_IDX[primary]] = 1.0 - self.label_smooth / 2

        return torch.from_numpy(wav), torch.from_numpy(label)


# Sanity check
ds_check = XCDataset(train_df.head(10), training=True)
wav, label = ds_check[0]
print(f"wav: {wav.shape} dtype={wav.dtype}, range=[{wav.min().item():.3f}, {wav.max().item():.3f}]")
print(f"label: sum={label.sum().item():.2f}, max={label.max().item():.3f}")


wav: torch.Size([160000]) dtype=torch.float32, range=[-0.133, 0.140]
label: sum=6.80, max=0.975


In [8]:
# ============================================================
# Cell 8: Model (SED with framewise attention)
# ============================================================
import torchaudio

class MelExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP,
            n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX,
        )
        self.db = torchaudio.transforms.AmplitudeToDB(top_db=80.0)

    def forward(self, wav):
        mel = self.mel(wav)
        mel = self.db(mel)
        mel = torch.clamp(mel, -80.0, 0.0)
        mel = (mel + 40.0) / 40.0
        return mel


class SEDHead(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.att = nn.Linear(in_dim, n_classes)
        self.cla = nn.Linear(in_dim, n_classes)

    def forward(self, x):
        att = torch.tanh(self.att(x))
        cla = self.cla(x)
        norm_att = F.softmax(att, dim=1)
        clipwise = (norm_att * cla).sum(dim=1)
        return clipwise, cla


class SEDModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            CFG.BACKBONE, pretrained=True, in_chans=3,
            num_classes=0, global_pool="",
        )
        feat_dim = self.backbone.num_features
        self.head = SEDHead(feat_dim, CFG.N_CLASSES)

    def forward(self, mel):
        x = mel.unsqueeze(1).repeat(1, 3, 1, 1)
        feat = self.backbone(x)
        feat = feat.mean(dim=2)
        feat = feat.transpose(1, 2)
        clipwise, framewise = self.head(feat)
        return clipwise, framewise


# Sanity check
model_check = SEDModel().to(DEVICE)
dummy = torch.randn(2, CFG.N_MELS, 313).to(DEVICE)
with torch.no_grad():
    out, fr = model_check(dummy)
print(f"clipwise: {out.shape}, framewise: {fr.shape}")
print(f"Model params: {sum(p.numel() for p in model_check.parameters())/1e6:.1f}M")
del model_check; torch.cuda.empty_cache()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/86.5M [00:00<?, ?B/s]

clipwise: torch.Size([2, 234]), framewise: torch.Size([2, 10, 234])
Model params: 20.8M


In [9]:
# ============================================================
# Cell 9: Augmentations
# ============================================================
def spec_mixup(mel, label, alpha=CFG.MIXUP_ALPHA, p=CFG.MIXUP_PROB):
    if np.random.random() > p:
        return mel, label
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(mel.size(0), device=mel.device)
    mel_mix = lam * mel + (1 - lam) * mel[idx]
    label_mix = torch.maximum(label, label[idx])
    return mel_mix, label_mix


def wave_mixup(wav, label, alpha=0.5, p=CFG.WAVE_MIXUP_PROB):
    """Waveform domain mixup (paper 256 流)."""
    if np.random.random() > p:
        return wav, label
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(wav.size(0), device=wav.device)
    wav_mix = lam * wav + (1 - lam) * wav[idx]
    label_mix = torch.maximum(label, label[idx])
    return wav_mix, label_mix


def spec_augment(mel, freq_mask=CFG.FREQ_MASK, time_mask=CFG.TIME_MASK):
    B, F_, T = mel.shape
    for b in range(B):
        if freq_mask > 0:
            f = np.random.randint(0, freq_mask)
            f0 = np.random.randint(0, max(1, F_ - f))
            mel[b, f0:f0+f, :] = 0
        if time_mask > 0:
            t = np.random.randint(0, time_mask)
            t0 = np.random.randint(0, max(1, T - t))
            mel[b, :, t0:t0+t] = 0
    return mel


In [10]:
# ============================================================
# Cell 10: Train/val split + DataLoaders
# ============================================================
train_df_s = train_df.sample(frac=1.0, random_state=CFG.SEED).reset_index(drop=True)
n_val = int(len(train_df_s) * 0.05)
train_split = train_df_s.iloc[n_val:].reset_index(drop=True)
val_split = train_df_s.iloc[:n_val].reset_index(drop=True)
print(f"train: {len(train_split)}, val: {len(val_split)}")

train_ds = XCDataset(train_split, training=True)
val_ds = XCDataset(val_split, training=False)

train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
                          num_workers=CFG.NUM_WORKERS, pin_memory=True,
                          drop_last=True, persistent_workers=True)
val_loader = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                         num_workers=CFG.NUM_WORKERS, pin_memory=True,
                         persistent_workers=True)
print(f"steps/ep: train={len(train_loader)}, val={len(val_loader)}")


train: 25059, val: 1318
steps/ep: train=97, val=6


In [11]:
# ============================================================
# Cell 11: Model + Optimizer + Scheduler
# ============================================================
model = SEDModel().to(DEVICE)
if torch.cuda.device_count() > 1:
    print(f"Using DataParallel on {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

mel_extractor = MelExtractor().to(DEVICE)

# AdamW with warmup
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS)
# scaler = GradScaler("cuda", init_scale=2**10)  # not needed with bf16
loss_fn = nn.BCEWithLogitsLoss()
print(f"Model params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
print(f"LR: {CFG.LR}, WD: {CFG.WD}, Warmup: {CFG.WARMUP_STEPS} steps")


Model params: 20.8M
LR: 0.0005, WD: 0.0001, Warmup: 500 steps


In [12]:
# ============================================================
# Cell 12: rich_evaluate (val_ns22 + val_macro + per-taxon AUC + class dist)
# ============================================================
from sklearn.metrics import roc_auc_score

@torch.no_grad()
def rich_evaluate(model, mel_ex, loader):
    model.eval()
    all_logits, all_labels = [], []
    for wav, label in tqdm(loader, desc="val", leave=False):
        wav = wav.to(DEVICE, non_blocking=True)
        mel = mel_ex(wav)
        with autocast("cuda", dtype=torch.bfloat16):
            logit, _ = model(mel)
        all_logits.append(logit.float().cpu())
        all_labels.append(label)
    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    # Threshold label > 0.5 (label smooth により 0.99-1.0 が positive)
    bin_labels = (labels > 0.5).astype(np.float32)

    per_class = []
    for c in range(bin_labels.shape[1]):
        n_pos = int(bin_labels[:, c].sum())
        if n_pos >= 2 and n_pos < bin_labels.shape[0]:
            try:
                auc = roc_auc_score(bin_labels[:, c], logits[:, c])
                per_class.append((PRIMARY_LABELS[c], float(auc), n_pos))
            except Exception:
                pass

    aucs_n22 = [auc for _, auc, n in per_class if n >= 22]
    val_ns22 = float(np.mean(aucs_n22)) if aucs_n22 else 0.0
    val_macro = float(np.mean([auc for _, auc, _ in per_class])) if per_class else 0.0

    taxon_aucs = {}
    for taxon in ["Aves", "Amphibia", "Insecta", "Mammalia", "Reptilia"]:
        aucs_in = [auc for label, auc, _ in per_class if LABEL_TO_CLASS.get(label) == taxon]
        taxon_aucs[taxon] = float(np.mean(aucs_in)) if aucs_in else float("nan")

    if aucs_n22:
        arr = np.array(aucs_n22)
        cs = {"n_valid": len(arr), "median": float(np.median(arr)),
              "p25": float(np.percentile(arr, 25)), "p75": float(np.percentile(arr, 75)),
              "n_above_05": int(np.sum(arr > 0.5)), "n_above_07": int(np.sum(arr > 0.7)),
              "n_above_09": int(np.sum(arr > 0.9)), "n_perfect": int(np.sum(arr >= 0.9999))}
    else:
        cs = {"n_valid": 0, "median": 0.0, "p25": 0.0, "p75": 0.0,
              "n_above_05": 0, "n_above_07": 0, "n_above_09": 0, "n_perfect": 0}

    return val_ns22, val_macro, taxon_aucs, cs


In [13]:
# ============================================================
# Cell 13: Training loop (30 ep, safe AMP, rich log)
# ============================================================
import sys
def _p(msg):
    print(msg, flush=True)
    sys.stdout.flush()

# Smoke test
_p(f"[smoke] iter + forward + backward...")
smoke_iter = iter(train_loader)
test_wav, test_label = next(smoke_iter)
test_wav_g = test_wav.to(DEVICE)
test_label_g = test_label.to(DEVICE)
test_mel = mel_extractor(test_wav_g)
_p(f"[smoke] mel: {test_mel.shape}, range=[{test_mel.min().item():.2f}, {test_mel.max().item():.2f}]")
with autocast("cuda", dtype=torch.bfloat16):
    test_logit, _ = model(test_mel)
    test_loss = loss_fn(test_logit, test_label_g)
_p(f"[smoke] loss={test_loss.item():.4f}")
if not (torch.isnan(test_loss) or torch.isinf(test_loss)):
    test_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    _p(f"[smoke] OK")
del smoke_iter, test_wav, test_label, test_wav_g, test_label_g, test_mel, test_logit, test_loss
torch.cuda.empty_cache()
_p(f"\n=== Training start ===\n")

history = {"train_loss": [], "val_ns22": [], "val_macro": [],
           "taxon_aucs": [], "class_stats": [], "lr": [], "elapsed_min": []}
best_val = 0.0
start_t = time.time()
nan_skip_count = 0
global_step = 0

for epoch in range(1, CFG.EPOCHS + 1):
    ep_start = time.time()
    _p(f"[ep{epoch}] start")
    model.train()
    train_losses = []
    pbar = tqdm(train_loader, desc=f"Ep {epoch}/{CFG.EPOCHS}", leave=False, file=sys.stdout)
    for batch_idx, (wav, label) in enumerate(pbar):
        # LR warmup
        if global_step < CFG.WARMUP_STEPS:
            lr_scale = (global_step + 1) / CFG.WARMUP_STEPS
            for g in optimizer.param_groups:
                g["lr"] = CFG.LR * lr_scale

        wav = wav.to(DEVICE, non_blocking=True)
        label = label.to(DEVICE, non_blocking=True)

        # Waveform mixup (before mel)
        wav, label = wave_mixup(wav, label)

        mel = mel_extractor(wav)
        # Spec mixup
        mel, label = spec_mixup(mel, label)
        mel = spec_augment(mel)

        with autocast("cuda", dtype=torch.bfloat16):
            logit, _ = model(mel)
            loss = loss_fn(logit, label)

        if torch.isnan(loss) or torch.isinf(loss):
            nan_skip_count += 1
            optimizer.zero_grad(set_to_none=True)
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        train_losses.append(loss.item())
        global_step += 1

        if batch_idx % 50 == 0:
            _p(f"[ep{epoch}] step {batch_idx}/{len(train_loader)} loss={np.mean(train_losses[-50:]):.4f} lr={optimizer.param_groups[0]['lr']:.2e}")
            torch.cuda.empty_cache()
        del mel, logit, loss

    if global_step >= CFG.WARMUP_STEPS:
        scheduler.step()
    torch.cuda.empty_cache()

    train_loss = float(np.mean(train_losses)) if train_losses else float("nan")
    _p(f"[ep{epoch}] train done (loss={train_loss:.4f}), val...")
    val_ns22, val_macro, taxon_aucs, class_stats = rich_evaluate(model, mel_extractor, val_loader)
    lr_now = optimizer.param_groups[0]["lr"]
    elapsed_min = (time.time() - start_t) / 60
    ep_min = (time.time() - ep_start) / 60

    history["train_loss"].append(train_loss)
    history["val_ns22"].append(val_ns22)
    history["val_macro"].append(val_macro)
    history["taxon_aucs"].append(taxon_aucs)
    history["class_stats"].append(class_stats)
    history["lr"].append(lr_now)
    history["elapsed_min"].append(elapsed_min)

    is_best = val_ns22 > best_val
    if is_best:
        best_val = val_ns22
        torch.save(
            {"epoch": epoch, "val_ns22": val_ns22, "val_macro": val_macro,
             "taxon_aucs": taxon_aucs, "class_stats": class_stats,
             "backbone": model.module.backbone.state_dict() if isinstance(model, nn.DataParallel) else model.backbone.state_dict(),
             "head": model.module.head.state_dict() if isinstance(model, nn.DataParallel) else model.head.state_dict(),
             "config": {k: str(v) for k, v in CFG.__dict__.items() if not k.startswith("_")},
            },
            CFG.BEST_CKPT,
        )

    json.dump(history, open(CFG.HIST_JSON, "w"), indent=2)
    taxon_str = " ".join(f"{t}={v:.3f}" for t, v in taxon_aucs.items())
    cs = class_stats
    _p(f"=== Ep {epoch}/{CFG.EPOCHS}: loss={train_loss:.4f} val_ns22={val_ns22:.4f} val_macro={val_macro:.4f}"
       f" {'BEST' if is_best else ''} lr={lr_now:.2e} ({ep_min:.1f}min, total {elapsed_min:.1f}min, nan_skips={nan_skip_count}) ===")
    _p(f"    taxon: {taxon_str}")
    _p(f"    class: n={cs['n_valid']} median={cs['median']:.3f} p25={cs['p25']:.3f} p75={cs['p75']:.3f}"
       f" #>0.5={cs['n_above_05']} #>0.7={cs['n_above_07']} #>0.9={cs['n_above_09']} #perfect={cs['n_perfect']}")

# Save final
torch.save(
    {"epoch": CFG.EPOCHS, "val_ns22": history["val_ns22"][-1],
     "backbone": model.module.backbone.state_dict() if isinstance(model, nn.DataParallel) else model.backbone.state_dict(),
     "head": model.module.head.state_dict() if isinstance(model, nn.DataParallel) else model.head.state_dict(),
    },
    CFG.LAST_CKPT,
)
_p(f"\n=== Stage 1 DONE. Best val_ns22={best_val:.4f} ===")
_p(f"Saved: {CFG.BEST_CKPT}")
_p(f"Saved: {CFG.LAST_CKPT}")


[smoke] iter + forward + backward...
[smoke] mel: torch.Size([256, 128, 313]), range=[0.28, 1.00]
[smoke] loss=0.7224
[smoke] OK

=== Training start ===

[ep1] start


Ep 1/30:   0%|          | 0/97 [00:00<?, ?it/s]

[ep1] step 0/97 loss=0.6052 lr=1.00e-06
[ep1] step 50/97 loss=0.4502 lr=5.10e-05
[ep1] train done (loss=0.3080), val...


val:   0%|          | 0/6 [00:00<?, ?it/s]

=== Ep 1/30: loss=0.3080 val_ns22=0.5116 val_macro=0.5100 BEST lr=9.70e-05 (0.7min, total 0.7min, nan_skips=0) ===
    taxon: Aves=0.510 Amphibia=nan Insecta=nan Mammalia=nan Reptilia=nan
    class: n=7 median=0.526 p25=0.455 p75=0.548 #>0.5=4 #>0.7=0 #>0.9=0 #perfect=0
[ep2] start


...

=== Ep 29/30: loss=0.1343 val_ns22=0.9401 val_macro=0.9497  lr=4.77e-05 (0.7min, total 20.9min, nan_skips=0) ===
    taxon: Aves=0.950 Amphibia=nan Insecta=nan Mammalia=nan Reptilia=nan
    class: n=7 median=0.954 p25=0.914 p75=0.976 #>0.5=7 #>0.7=7 #>0.9=5 #perfect=0
[ep30] start


Ep 30/30:   0%|          | 0/97 [00:00<?, ?it/s]

[ep30] step 0/97 loss=0.1345 lr=4.77e-05
[ep30] step 50/97 loss=0.1353 lr=4.77e-05
[ep30] train done (loss=0.1354), val...


val:   0%|          | 0/6 [00:00<?, ?it/s]

=== Ep 30/30: loss=0.1354 val_ns22=0.9369 val_macro=0.9500  lr=3.35e-05 (0.7min, total 21.6min, nan_skips=0) ===
    taxon: Aves=0.950 Amphibia=nan Insecta=nan Mammalia=nan Reptilia=nan
    class: n=7 median=0.952 p25=0.915 p75=0.975 #>0.5=7 #>0.7=7 #>0.9=5 #perfect=0

=== Stage 1 DONE. Best val_ns22=0.9423 ===
Saved: /content/drive/MyDrive/kaggle/birdclef2026/exp052/ckpt/stage1_backbone_best.pth
Saved: /content/drive/MyDrive/kaggle/birdclef2026/exp052/ckpt/stage1_backbone_last.pth


In [14]:
# ============================================================
# Cell 15: Upload ckpt as Kaggle Dataset
# ============================================================
import json, shutil
from pathlib import Path

UPLOAD_DIR = Path("/content/exp052_upload")
UPLOAD_DIR.mkdir(exist_ok=True, parents=True)

# Copy ckpts + history
for src_file in [CFG.BEST_CKPT, CFG.LAST_CKPT, CFG.HIST_JSON]:
    if src_file.exists():
        shutil.copy(src_file, UPLOAD_DIR / src_file.name)
        print(f"  Copied: {src_file.name} ({src_file.stat().st_size/1e6:.1f} MB)")
    else:
        print(f"  WARN: {src_file.name} not found")

# Dataset metadata
USER = "maekeso"
SLUG = "birdclef2026-exp052-stage1-effv2s"
meta = {
    "title": "BirdCLEF2026 exp052 Stage 1 EfficientNetV2-S",
    "id": f"{USER}/{SLUG}",
    "licenses": [{"name": "other"}],
}
(UPLOAD_DIR / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))

# Try version up first (if Dataset exists), fall back to create new
import subprocess

def run(cmd):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print(f"stderr: {r.stderr[:500]}")
    return r.returncode

# Try version up
ret = run(f"kaggle datasets version -p {UPLOAD_DIR} -m 'Stage 1 epoch {CFG.EPOCHS} best={best_val:.4f}' -r tar")
if ret != 0:
    # Fall back: create new
    print("Version up failed, trying create new...")
    ret = run(f"kaggle datasets create -p {UPLOAD_DIR} -r tar")

if ret == 0:
    print(f"\nOK Dataset uploaded: https://www.kaggle.com/datasets/{USER}/{SLUG}")
else:
    print(f"\nERR upload failed. Manual upload:")
    print(f"  files: {[f.name for f in UPLOAD_DIR.iterdir()]}")


  Copied: stage1_backbone_best.pth (84.0 MB)
  Copied: stage1_backbone_last.pth (84.0 MB)
  Copied: stage1_history.json (0.0 MB)
$ kaggle datasets version -p /content/exp052_upload -m 'Stage 1 epoch 30 best=0.9423' -r tar
Starting upload for file stage1_backbone_last.pth
Upload successful: stage1_backbone_last.pth (80MB)
Starting upload for file stage1_backbone_best.pth
Upload successful: stage1_backbone_best.pth (80MB)
Starting upload for file stage1_history.json
Upload successful: stage1_history.json (14KB)
403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/CreateDatasetVersion

stderr: 
  0%|          | 0.00/80.1M [00:00<?, ?B/s]
  1%|▏         | 1.02M/80.1M [00:00<00:08, 9.35MB/s]
  2%|▏         | 1.92M/80.1M [00:00<00:30, 2.72MB/s]
  5%|▌         | 4.17M/80.1M [00:00<00:13, 6.12MB/s]
  8%|▊         | 6.64M/80.1M [00:00<00:08, 9.48MB/s]
 17%|█▋        | 13.3M/80.1M [00:01<00:03, 21.3MB/s]
 23%|██▎       | 18.4M/80.1M [00:01<00:02, 27.5MB/s]
 2

In [15]:
# ============================================================
# Cell 14: Summary + transfer prep
# ============================================================
print(f"=== Stage 1 Summary ===")
print(f"  Backbone: {CFG.BACKBONE}")
print(f"  Epochs trained: {CFG.EPOCHS}")
print(f"  Best val_ns22: {best_val:.4f}")
print(f"  Total time: {(time.time() - start_t)/60:.1f} min")
print(f"  NaN skips: {nan_skip_count}")
print()
print(f"Best ckpt path: {CFG.BEST_CKPT}")
print(f"  size: {CFG.BEST_CKPT.stat().st_size/1e6:.1f} MB")
print()
print(f"Next steps (Stage 2):")
print(f"  1. Load backbone from {CFG.BEST_CKPT}['backbone']")
print(f"  2. Re-init head for BC2026 finetune (head_reinit=True)")
print(f"  3. Train on BC2026 train_audio for 20-30 epoch")
print(f"  4. Expected LB +0.008-0.025")


=== Stage 1 Summary ===
  Backbone: tf_efficientnetv2_s.in21k_ft_in1k
  Epochs trained: 30
  Best val_ns22: 0.9423
  Total time: 22.0 min
  NaN skips: 0

Best ckpt path: /content/drive/MyDrive/kaggle/birdclef2026/exp052/ckpt/stage1_backbone_best.pth
  size: 84.0 MB

Next steps (Stage 2):
  1. Load backbone from /content/drive/MyDrive/kaggle/birdclef2026/exp052/ckpt/stage1_backbone_best.pth['backbone']
  2. Re-init head for BC2026 finetune (head_reinit=True)
  3. Train on BC2026 train_audio for 20-30 epoch
  4. Expected LB +0.008-0.025


In [16]:
# ============================================================
# Cell 16: Disconnect runtime (auto-shutdown to save Colab credits)
# ============================================================
print("All done. Disconnecting runtime in 30 sec to save Colab credits...")
print("(if you want to keep runtime, interrupt this cell within 30 sec)")

import time
time.sleep(30)

from google.colab import runtime
runtime.unassign()  # auto-disconnect


All done. Disconnecting runtime in 30 sec to save Colab credits...
(if you want to keep runtime, interrupt this cell within 30 sec)
